# OAI with Physical Radio Unit in R2LAB

## Prerequisites

Before starting, make sure you:

- Have completed the [Provision Resources](./01_provision.ipynb), [RAN split](./22_split.ipynb), and [TDD patterns](./20_tdd.ipynb) tutorials.
- Have a basic understanding of Python.
- Are familiar with 5G architecture and 5G NR radio.

## Tutorial Overview

In this tutorial, you will deploy a real Radio Unit (RU) inside the [R2LAB anechoic chamber](https://r2lab.inria.fr/index.md) at SophiaNode. 

### What You Will Build

The deployment consists of:
* **Core Network:** Open5GS core running on a virtual machine (VM).
* **Ran Deployment:** An OpenAirInterface (OAI) RAN split across a CU-CP, CU-UP, and DU:
  * A VM hosting the CU-CP and CU-UP functions.
  * A dedicated bare-metal server (`sopnode-f1`) hosting the DU and the 5G fronthaul connection.
* **Radio Unit (RU):** An [AW2S Jaguar](https://www.aw2s.com/solutions/rrh-rru/) RU hosted inside the chamber, connected to the DU over fiber.
* **User Equipment (UE):** Two physical 5G Quectel RM500Q-GL UEs (`qhat01` and `qhat02`), each attached to a Raspberry Pi 4 and connected to the RU.

### Architecture & Infrastructure

While previous tutorials relied entirely on virtual machines, real RU hardware requires a bare-metal DU to deliver the low-latency performance needed for radio signal processing. 

To support this setup, SophiaNode provides high-performance 5G fronthaul servers directly linked to RUs over fiber, alongside SLICES-BI virtual machines. In this scenario, you will combine both infrastructure types:

| Component | Target Environment | Hardware / Machine |
| :--- | :--- | :--- |
| **5G Core** | Virtual Machine | SLICES-BI VM |
| **CU-CP & CU-UP** | Virtual Machine | SLICES-BI VM |
| **DU & Fronthaul** | Bare-Metal | POS node (`sopnode-f1`) |

![RU topology](figures/ru.png)

> **Note:** While splitting the CU-CP, CU-UP, and DU is not strictly required for this physical RU setup, it is used here to demonstrate a realistic split-RAN deployment.

## Metamodel and Deployment Model

### Metamodel

The metamodel follows the same pattern used in previous tutorials, with one key addition: it can describe a physical Radio Unit (RU). 
* **Simulated Link:** If no RU is specified, the radio link is simulated.
* **Physical Link:** If an RU is specified, the gNB (or the DU in a split deployment) uses it for real-world radio transmission and reception.

#### Radio Unit (RU) Configuration

This tutorial uses the `jaguar` RU inside the [R2LAB anechoic chamber](https://r2lab.inria.fr/index.md) at SophiaNode. R2LAB is an open-access, remotely accessible wireless testbed providing software-defined radios (USRPs), 5G RUs, and physical UEs. You can check the [SophiaNode Resources page](https://gitlab.inria.fr/slices-ri/resources/sites/sophianode/resources) for full equipment details.

The [`Jaguar`](https://www.aw2s.com/solutions/rrh-rru/) is an AW2S 5G RU configured with the following parameters:
* **Frequency Band:** `n78`
* **Bandwidth:** 50 MHz
* **MIMO Configuration:** 2x2 MIMO
* **TDD Pattern:** `DDSUU` with a 2.5 ms period

> **Note:** Transmit attenuation is tuned specifically for the R2LAB environment. Examples of gNB configurations for R2LAB equipment are available [in this GitHub repository](https://github.com/sopnode/oai5g-rru/tree/main/rru).

#### User Equipment (UE) Configuration

The chamber houses several types of UEs, including smartphones and Quectel modules. This tutorial connects two physical Quectel RM500Q-GL devices named `qhat01` and `qhat02`. 

Because these are physical devices rather than software-emulated UEs (such as UERANSIM), the metamodel classifies them as **commercial off-the-shelf (COTS)** equipment. Their SUPI and authentication data are listed on the [R2LAB hardware page](https://r2lab.inria.fr/hardware.md).

In [ ]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  jaguar:
    type: real
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

ran:
    components:
      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          radioConfiguration:
            nrBand: 78
            channelBandwidthMhz: 50
            pointAFrequencyArfcn: 626748
            ssbArfcn: 627360
            ssbPbchBlockPowerDbm: -10
            tddConfiguration:
              periodicityMilliseconds: 2.5
              nrofDownlinkSlots: 2
              nrofDownlinkSymbols: 6
              nrofUplinkSlots: 2
              nrofUplinkSymbols: 4
          ru: jaguar
          cu_cp: cucp
          cu_up: cuup

ues:
  qhat01:
    type: cots
    config:
      supi: "001010000000006"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1

  qhat02:
    type: cots
    config:
      supi: "001010000000007"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1
""")

### Deployment Model

This tutorial combines several resource types: two VMs in SLICES-BI, one bare-metal machine in POS, one RU in R2LAB, and two physical UEs in R2LAB. These resources are managed by three distinct provisioners:

* **`bi` Provisioner:** Manages the VMs using the `fr-sophia2-bi-vm1` infrastructure, which is directly connected to R2LAB.
* **`legacy_pos` Provisioner:** Manages the POS bare-metal machine.
* **`r2lab` Provisioner:** Manages the RU and UEs hosted in R2LAB.

> **Important:** Reserve your R2LAB and POS resources *before* running this tutorial. While provisioners can attempt automatic reservations, this tutorial assumes your reservations are already active.

> **Important:** Never publish or share your R2LAB credentials.

#### Node Roles and Provisioning Parameters

* **Fronthaul Tuning:** The fronthaul machine requires performance tuning to interface correctly with the RU. This tutorial configures the Linux boot parameters for `sopnode-f1` via the node's `provisioning_params.boot_params` field.
* **RAN Distribution:** The RAN spans two machines, both assigned the `ran` role. The `node-ran` machine also carries the `cu_cp` and `cu_up` roles to host those respective functions.
* **RU & UE References:** The RU and UEs simply need to reference the `r2lab` provisioner.

#### Scheduling and Network Configuration

The CU-CP, CU-UP, and DU require specific scheduling constraints:
* Set `node_name` on the DU component to explicitly place it on `sopnode-f1`.
* Set `force_type_scheduling` on the CU-CP and CU-UP components to place each on a node with the corresponding `cu_cp` or `cu_up` role.

To establish the connection between the RU and the DU:
* Enable **Multus**, specifying the host network interface and local IP address designated for the RU connection.
* Configure OAI with Jaguar-specific settings by providing them in the DU component's `advanced_config` section.

For more details regarding SophiaNode equipment and network topology, see the [SophiaNode Resources page](https://gitlab.inria.fr/slices-ri/resources/sites/sophianode/resources).

A good practice to prevent private information leak (e.g., password) is to store the sensitive information in a special location. In this, we keep it as environment variables that we store in a [`.env` file](https://pypi.org/project/python-dotenv/). The following lines load variables from this `.env` file. Make sure to never commit your secrets in public repositories.

In [ ]:
!pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
R2LAB_SLICE=os.getenv("R2LAB_SLICE")
R2LAB_EMAIL=os.getenv("R2LAB_EMAIL")
R2LAB_PASSWORD=os.getenv("R2LAB_PASSWORD")

We can now create the deployment model.

In [ ]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
f"""
provisioners:
  bi-provisioner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: ru_tutorial
      create_experiment: true
      experiment_duration: 2h
      node_duration: 2h

  pos:
    type: pos
    params:
      attempt_reservation: false

  r2lab:
    type: r2lab
    params:
      r2lab_slice: {R2LAB_SLICE}
      reservation_params:
        email: {R2LAB_EMAIL}
        password: {R2LAB_PASSWORD}

nodes:
  sopnode-f1:
    provisioner: pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  node-core:
    provisioner: bi-provisioner
    roles:
    - core

  node-ran:
    provisioner: bi-provisioner
    roles:
    - ran
    - cu_cp
    - cu_up

rus:
  jaguar:
    vendor: aw2s
    model: jaguar
    provisioner: r2lab

ues:
  qhat01:
    type: cots
    provisioner: r2lab

  qhat02:
    type: cots
    provisioner: r2lab

ran:
  implementation: oai
  components:
    cucp:
      type: cu_cp
      force_type_scheduling: true

    cuup:
      type: cu_up
      force_type_scheduling: true

    du1:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.102"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 1
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f1
"""
)


## Provision the Infrastructure

First, provision only the compute nodes (the virtual machines and bare-metal servers).

To ensure the physical RUs and UEs are not provisioned yet, use the `provision_nodes` function rather than `provision`. Once all compute nodes are up and ready, proceed to deploy the Kubernetes cluster.

In [ ]:
from ifran import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(quiet=False)             

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

In [ ]:
from pathlib import Path
DIR="state/ru"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

The infrastructure is now ready for the 5G network deployment.

## Deploy the 5G network

### Deploy the 5G core

With the infrastructure provisioned, start by deploying the 5G core.

In [ ]:
from ifran import CoreManagement

core = CoreManagement(preceding=k8s)
await core.deploy()

## Tune the RAN Nodes

The Distributed Unit (DU) requires additional CPU tuning and network-interface configuration to meet connectivity and performance demands of the Radio Unit (RU). Because these settings depend on your specific RU/DU pair, they are applied directly to the machine via an **Ansible playbook**. 

The **`Miscellaneous`** manager is used to run arbitrary tasks as long as they are implemented in the form of [Ansible playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks_intro.html). The `Miscellaneous` manager executes these playbooks using the inventory automatically generated from your deployment model.

### Working with the Inventory and Playbook

* **Inspect the Inventory:** To review the generated inventory before applying changes, call `get_ansible_inventory` on the `ProvisioningManagement` manager.
* **Run the Playbook:** This tutorial uses a custom playbook tailored specifically for this hardware setup: [`misc/ansible/playbooks/tuning.yaml`](misc/ansible/playbooks/tuning.yaml).

In [ ]:
print (prov.get_ansible_inventory())

In [ ]:
from ifran import MiscellaneousManagement

misc = MiscellaneousManagement(preceding=core)

In [ ]:
await misc.run_playbook(
    playbook="./misc/ansible/playbooks/tuning.yaml",
    roles_paths=["./misc/ansible/roles"],
)

### Deploy the 5G RAN

The `sopnode-f1` machine is now configured. Provision the `jaguar` RU before deploying the DU.

In [ ]:
prov_ru = ProvisioningManagement(preceding=misc)
await prov_ru.provision()

With the radio hardware provisioned, deploy the RAN components. The CU-CP and CU-UP will run on `node-ran`, and the DU will run on `sopnode-f1`.

In [ ]:
from ifran import RANManagement

ran = RANManagement(preceding=prov_ru)
await ran.deploy()

Check the pods in the `oai-ran` namespace to verify that the RAN functions are running on the expected machines.

In [ ]:
print((await prov_ru.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov_ru.deployment_model.params.k8s.master_node],
    become=True, quiet=True, flatten=False))[prov_ru.deployment_model.params.k8s.master_node])

Check the DU logs to confirm that it is running.

In [ ]:
print ((await ran.get_logs(quiet=True, flatten=False))["du1"])

## Connect the UEs

With the 5G core network and both RAN deployments up and running, you can now provision the UEs defined in your metamodel.

For the physical R2LAB UEs, provisioning simply powers on the devices. Once active, attach them to their respective radio networks by calling `connect` on the `UEManagement` manager.

In [ ]:
prov_ues = ProvisioningManagement(preceding=ran)

await prov_ues.provision()

In [ ]:
from ifran import UEManagement

ues = UEManagement(preceding=prov_ues)
await ues.connect()

Check the DU logs to confirm that the UEs are connected through the radio interface.

In [ ]:
print (await ran.get_logs(quiet=True))

To test connectivity, send a ping from each UE over its radio interface. On the QHAT devices, this interface is named `wwan0`.

Run commands directly on the UEs with the `run_ue_command` function.

In [ ]:
prov_ues = ProvisioningManagement(preceding=ues)
command = ["ping", "-I", "wwan0", "-c", "10", "-D", "8.8.8.8"]
pings = await prov_ues.run_ue_command(command=command, ues=["qhat01", "qhat02"], quiet=True, flatten=False)

Ping results from `qhat01`:

In [ ]:
print (pings["qhat01"])

Ping results from `qhat02`:

In [ ]:
print (pings["qhat02"])

Finally, save the metamodel and deployment model to keep a record of this experiment.

In [ ]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")